# Day 4-02｜MediaPipe 人體姿態與投籃角度分析

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：用 MediaPipe 或合成資料取得肩膀、手肘、手腕、髖、膝、踝等點，計算 elbow / knee / shoulder angle。

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
# 第一次使用 MediaPipe 時再打開安裝。
# !pip install -q mediapipe

In [ ]:
import pandas as pd
from pathlib import Path
from src.video_utils import list_videos, pick_first_converted_video
from src.shooting_utils import synthetic_pose_sequence, add_pose_angles, draw_skeleton
from src.cv_utils import show_image, save_image_rgb
from src.plot_utils import plot_angle_series

converted = list_videos(COURSE_ROOT / "assets" / "converted")
print("converted videos:", [p.name for p in converted])

In [ ]:
# 初學保底：沒有影片或 MediaPipe 無法跑時，先使用合成 pose sequence。
use_synthetic = True
pose_df = synthetic_pose_sequence(n=80)
pose_df.head()

In [ ]:
# 選用：用 MediaPipe 讀學生影片。
# 這段保留成教學骨架，老師可課堂帶著跑。
RUN_MEDIAPIPE = False

if RUN_MEDIAPIPE:
    import cv2
    import mediapipe as mp
    from src.video_utils import pick_first_converted_video

    video_path = pick_first_converted_video(COURSE_ROOT)
    cap = cv2.VideoCapture(str(video_path))
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=False, model_complexity=1)

    rows = []
    frame_idx = 0
    stride = 5
    while cap.isOpened():
        ok, frame_bgr = cap.read()
        if not ok:
            break
        if frame_idx % stride != 0:
            frame_idx += 1
            continue
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        res = pose.process(frame_rgb)
        h, w = frame_rgb.shape[:2]
        if res.pose_landmarks:
            lm = res.pose_landmarks.landmark
            # 右手側；若拍攝方向相反，可以改成 LEFT_*。
            ids = mp_pose.PoseLandmark

            def pt(name):
                p = lm[getattr(ids, name).value]
                return p.x * w, p.y * h

            shoulder = pt("RIGHT_SHOULDER")
            elbow = pt("RIGHT_ELBOW")
            wrist = pt("RIGHT_WRIST")
            hip = pt("RIGHT_HIP")
            knee = pt("RIGHT_KNEE")
            ankle = pt("RIGHT_ANKLE")
            rows.append(
                {
                    "frame": frame_idx,
                    "shoulder_x": shoulder[0],
                    "shoulder_y": shoulder[1],
                    "elbow_x": elbow[0],
                    "elbow_y": elbow[1],
                    "wrist_x": wrist[0],
                    "wrist_y": wrist[1],
                    "hip_x": hip[0],
                    "hip_y": hip[1],
                    "knee_x": knee[0],
                    "knee_y": knee[1],
                    "ankle_x": ankle[0],
                    "ankle_y": ankle[1],
                }
            )
        frame_idx += 1
    cap.release()
    pose.close()
    pose_df = add_pose_angles(pd.DataFrame(rows))
    use_synthetic = False

print("rows:", len(pose_df), "synthetic:", use_synthetic)
pose_df.head()

In [ ]:
out_csv = COURSE_ROOT / "assets" / "results" / "d4_02_pose_angles.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
pose_df.to_csv(out_csv, index=False)
print("saved:", out_csv)

plot_angle_series(
    pose_df,
    ["elbow_angle", "knee_angle", "shoulder_angle"],
    output_path=COURSE_ROOT / "assets" / "results" / "d4_02_pose_angle_plot.png",
)

In [ ]:
# 顯示其中一個 frame 的骨架圖。
row = pose_df.iloc[len(pose_df) // 2]
img = draw_skeleton(640, 480, row)
show_image(img, "pose skeleton sample")
save_image_rgb(
    COURSE_ROOT / "assets" / "results" / "d4_02_pose_skeleton_sample.png", img
)

小練習：如果學生是左手投籃，MediaPipe 區塊要把 `RIGHT_*` 改成 `LEFT_*`。